# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [2]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [1]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [11]:

import pandas as pd

# URL del dataset de salarios de Chicago
url = "https://data.cityofchicago.org/api/views/xzkq-xp2w/rows.csv?accessType=DOWNLOAD"

# Cargar directamente desde internet
salaries = pd.read_csv(url)

# Verificar que se cargó
print("Dataset loaded successfully!")
print(f"Shape: {salaries.shape}")
salaries.head()

Dataset loaded successfully!
Shape: (31824, 8)


,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"SANFRATELLO, VINCENT A",BRICKLAYER,DEPARTMENT OF WATER MANAGEMENT,F,HOURLY,40.0,NaN,53.06
1,"TILLMAN, NELSON P",SANITATION LABORER,DEPARTMENT OF STREETS AND SANITATION,F,HOURLY,40.0,NaN,47.79
2,"WALL, TAHARI L",CONSTRUCTION LABORER,DEPARTMENT OF WATER MANAGEMENT,F,HOURLY,40.0,NaN,51.40
3,"HEHIR, CONOR R",ASST CHIEF OPERATING ENGINEER,DEPARTMENT OF WATER MANAGEMENT,F,HOURLY,40.0,NaN,68.26
4,"BURFORD, JAMES L",CEMENT FINISHER,DEPARTMENT OF WATER MANAGEMENT,F,HOURLY,40.0,NaN,53.10


Examine the `salaries` dataset using the `head` function below.

In [13]:
if 'Hourly Rate' in salaries.columns:
    hourly_wages = salaries[salaries['Hourly Rate'].notna()]['Hourly Rate']
else:
    # Alternative: look for salary type column
    if 'Salary or Hourly' in salaries.columns:
        hourly_wages = salaries[salaries['Salary or Hourly'] == 'Hourly']['Hourly Rate'].dropna()
    else:
        hourly_wages = salaries[salaries['Typical Hours'].notna()]['Hourly Rate'].dropna()

print(f"\nNumber of hourly workers: {len(hourly_wages)}")
print(f"Mean hourly wage: ${hourly_wages.mean():.2f}")


Number of hourly workers: 6976
Mean hourly wage: $46.29


# Challenge 2 - Hypothesis Tests

In this section of the lab, we will test whether the hourly wage of all hourly workers is significantly different from $30/hr. Import the correct one sample test function from scipy and perform the hypothesis test for a 95% two sided confidence interval.

In [19]:
# Importar la función necesaria
from scipy.stats import ttest_1samp

In [21]:
t_stat, p_value = ttest_1samp(hourly_wages, 30)
print(f"T-test (H0: mean = $30)")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Significant difference from $30: {'YES' if p_value < 0.05 else 'NO'}")

T-test (H0: mean = $30)
T-statistic: 94.6778
P-value: 0.000000
Significant difference from $30: YES


# Challenge 3 - Constructing Confidence Intervals

While testing our hypothesis is a great way to gather empirical evidence for accepting or rejecting the hypothesis, another way to gather evidence is by creating a confidence interval. A confidence interval gives us information about the true mean of the population. So for a 95% confidence interval, we are 95% sure that the mean of the population is within the confidence interval. 
).

To read more about confidence intervals, click [here](https://en.wikipedia.org/wiki/Confidence_interval).


In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. 

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter and the standard error for the scale. The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [16]:
ci = t.interval(0.95, len(hourly_wages)-1, loc=hourly_wages.mean(), scale=sem(hourly_wages))
print(f"95% Confidence Interval: (${ci[0]:.2f}, ${ci[1]:.2f})")



95% Confidence Interval: ($45.95, $46.63)


# Challenge 4 - Hypothesis Tests of Proportions

Another type of one sample test is a hypothesis test of proportions. In this test, we examine whether the proportion of a group in our sample is significantly different than a fraction. 

You can read more about one sample proportion tests [here](http://sphweb.bumc.bu.edu/otlt/MPH-Modules/BS/SAS/SAS6-CategoricalData/SAS6-CategoricalData2.html).

In the cell below, use the `proportions_ztest` function from `statsmodels` to perform a hypothesis test that will determine whether the number of hourly workers in the City of Chicago is significantly different from 25% at the 95% confidence level.

In [18]:
total = len(salaries)
hourly_count = len(hourly_wages)
p_hat = hourly_count / total
p_0 = 0.25

z = (p_hat - p_0) / np.sqrt((p_0 * (1 - p_0)) / total)
p_prop = 2 * (1 - norm.cdf(abs(z)))

print(f"Proportion of hourly workers: {p_hat:.4f} ({p_hat:.2%})")
print(f"Hypothesized proportion: {p_0:.2%}")
print(f"Z-statistic: {z:.4f}")
print(f"P-value: {p_prop:.6f}")
print(f"Significantly different from 25%: {'YES' if p_prop < 0.05 else 'NO'}")

Proportion of hourly workers: 0.2192 (21.92%)
Hypothesized proportion: 25.00%
Z-statistic: -12.6867
P-value: 0.000000
Significantly different from 25%: YES
